# Exercise 6.1: The RAG Pipeline on Real PDFs (Research Papers)

We start from raw PDFs and own the whole path: **ingest → clean → chunk → index → retrieve (BM25, dense, hybrid, HyDE) → answer with citations**.

| File | Paper | Notes |
|---|---|---|
| `attention.pdf` | Attention Is All You Need | |
| `bert.pdf` | BERT | |
| `gpt2.pdf` | Language Models are Few-Shot Learners (GPT-3) | file is misnamed |

**Metric.** `hit@k` = the reference answer appears (after `rag_common.normalize_text`) in at least one of the top-k retrieved chunks. It does not depend on how we chunk, so chunkings can be compared fairly.

**Code reuse.** Ingestion is from 6.2, the splitter from 6.6, BM25 and `reciprocal_rank_fusion` from 6.7, the HyDE prompt from 6.8, and the cited-answer prompt and parser from 6.13.

**Running on Colab.** A GPU runtime is best. The notebook mounts Google Drive and saves to `MyDrive/ADS-NLP/Week6/ex6_results/`, as in the loss-mask exercise:
- chunk embeddings (`emb_*.npy`)
- every LLM reply (`llm_cache.json`)
- the result tables

After a disconnect, rerun everything. Cached embeddings and LLM replies are loaded, not recomputed.

---
## 0. Setup

In [ ]:
import importlib.util, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
REQS = {"pymupdf": "pymupdf", "rank_bm25": "rank_bm25", "langchain_text_splitters": "langchain-text-splitters",
        "langchain_openai": "langchain-openai", "sentence_transformers": "sentence-transformers", "yaml": "pyyaml", "tabulate": "tabulate"}
missing = [pip for mod, pip in REQS.items() if importlib.util.find_spec(mod) is None]
if missing:   # e.g. a fresh Colab runtime
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import bisect, glob, hashlib, json, os, re, shutil, time, urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf
import torch

pd.set_option("display.width", 200, "display.max_colwidth", 80)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Colab: {IN_COLAB} | device: {device}")

### Find the PDFs (Google Drive) and set the results folder

We look for `data/papers/` in Drive, next to this notebook and in `/content`. If the PDFs are not there, we download them from arXiv as 6.2 does.

To reuse the local copies, upload the Week6 `data/papers/` folder to `MyDrive/ADS-NLP/Week6/data/papers/`. Also upload `rag_common.py` and `config.yaml` to `MyDrive/ADS-NLP/Week6/` if you have them. Without them, the notebook uses an exact copy of `normalize_text` and the default embedding model.

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/ADS-NLP/Week6")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BASE = DRIVE_ROOT if IN_COLAB else Path(".")

PAPERS = {   # file -> arXiv URL (from 6.2)
    "attention.pdf": "https://arxiv.org/pdf/1706.03762",
    "bert.pdf":      "https://arxiv.org/pdf/1810.04805",
    "gpt2.pdf":      "https://arxiv.org/pdf/2005.14165",   # GPT-3 paper, misnamed
}
SEARCH = [BASE / "data/papers", Path("data/papers"), Path("/content/data/papers"), Path("../Week6/data/papers")]

def find_papers():
    for d in SEARCH:
        if all((d / f).is_file() for f in PAPERS):
            return d
    for depth in range(5) if IN_COLAB else []:   # search Drive a few levels deep
        for hit in glob.glob("/content/drive/MyDrive/" + "*/" * depth + "papers/attention.pdf"):
            d = Path(hit).parent
            if all((d / f).is_file() for f in PAPERS):
                return d
    return None

PAPER_DIR = find_papers()
if PAPER_DIR is None:   # first run: download from arXiv (as in 6.2)
    import requests
    PAPER_DIR = BASE / "data/papers"
    PAPER_DIR.mkdir(parents=True, exist_ok=True)
    for name, url in PAPERS.items():
        print("Downloading", name)
        (PAPER_DIR / name).write_bytes(requests.get(url, timeout=60).content)

RESULTS_DIR = BASE / "ex6_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("PDFs   :", PAPER_DIR.resolve(), sorted(p.name for p in PAPER_DIR.glob("*.pdf")))
print("Results:", RESULTS_DIR.resolve())

In [ ]:
# normalize_text and token_f1 come from rag_common.py; the fallbacks are exact copies.
sys.path.insert(0, str(BASE))
try:
    import rag_common as rc
    normalize_text, token_f1 = rc.normalize_text, rc.token_f1
    print("Using rag_common.py")
except Exception:
    def normalize_text(text: str) -> str:
        return " ".join(re.sub(r"[^a-z0-9\s]", " ", text.lower()).split())

    def token_f1(prediction: str, reference: str) -> float:
        pred, ref = normalize_text(prediction).split(), normalize_text(reference).split()
        overlap = sum((Counter(pred) & Counter(ref)).values()) if pred and ref else 0
        if overlap == 0:
            return 0.0
        precision, recall = overlap / len(pred), overlap / len(ref)
        return 2 * precision * recall / (precision + recall)
    print("rag_common.py not found: using the copied helpers")

try:
    import yaml
    DENSE_MODEL = yaml.safe_load((BASE / "config.yaml").read_text())["embeddings"]["dense_model"]
except Exception:
    DENSE_MODEL = "BAAI/bge-small-en-v1.5"   # embeddings.dense_model in config.yaml
print("Dense model:", DENSE_MODEL)

### The questions

The 12 given questions, plus **5 of our own** (`own=True`). Each of ours has a short answer that appears in the PDF text; this is checked in Part 1.

In [ ]:
questions = [
    {"question": "What BLEU score did the Transformer achieve on WMT 2014 English-to-German?", "answer": "28.4", "paper": "attention.pdf"},
    {"question": "What function turns the scaled dot products into attention weights?", "answer": "softmax", "paper": "attention.pdf"},
    {"question": "Why does multi-head attention project queries, keys and values several times?", "answer": "different representation subspaces", "paper": "attention.pdf"},
    {"question": "What functions define the positional encodings?", "answer": "sine and cosine", "paper": "attention.pdf"},
    {"question": "When is self-attention faster than a recurrent layer?", "answer": "faster than recurrent layers", "paper": "attention.pdf"},
    {"question": "How often is a selected token replaced by the [MASK] token in BERT?", "answer": "80% of the time", "paper": "bert.pdf"},
    {"question": "How does BERT describe its sentence-pair pre-training task?", "answer": "binarized", "paper": "bert.pdf"},
    {"question": "How are the parameters treated when fine-tuning BERT on a downstream task?", "answer": "end-to-end", "paper": "bert.pdf"},
    {"question": "Which earlier model is the example of a feature-based approach in the BERT paper?", "answer": "ELMo", "paper": "bert.pdf"},
    {"question": "Does few-shot learning in GPT-3 update the model weights?", "answer": "gradient updates", "paper": "gpt2.pdf"},
    {"question": "How was the Common Crawl data deduplicated?", "answer": "fuzzy deduplication", "paper": "gpt2.pdf"},
    {"question": "How does model size relate to the learning rate used?", "answer": "smaller learning rate", "paper": "gpt2.pdf"},
]
own_questions = [
    {"question": "How many identical layers make up the Transformer encoder stack?", "answer": "N = 6", "paper": "attention.pdf"},
    {"question": "How large is the WordPiece vocabulary BERT uses?", "answer": "30,000 token vocabulary", "paper": "bert.pdf"},
    {"question": "How many words does the BooksCorpus used to pre-train BERT contain?", "answer": "800M words", "paper": "bert.pdf"},
    {"question": "What context window do all GPT-3 models use?", "answer": "2048 tokens", "paper": "gpt2.pdf"},
    {"question": "How many parameters does GPT-3 have?", "answer": "175 billion parameters", "paper": "gpt2.pdf"},
]
for q in questions:
    q["own"] = False
for q in own_questions:
    q["own"] = True
QS = questions + own_questions
for i, q in enumerate(QS, start=1):
    q["qid"] = f"Q{i:02d}"
print(f"{len(questions)} given + {len(own_questions)} own = {len(QS)} questions")
print(pd.Series([q["paper"] for q in QS]).value_counts().to_string())

---
# Part 1: Ingest, chunk, index

## 1.1 Ingest

PyMuPDF `page.get_text()` per page (6.2). Each page keeps its `source` file and `page` number. We use plain text rather than `pymupdf4llm` Markdown here because it shows the raw extraction problems we want to measure.

In [ ]:
def ingest(pdf_dir: Path) -> list[dict]:
    pages = []
    for name in PAPERS:
        with pymupdf.open(pdf_dir / name) as doc:
            for i, page in enumerate(doc):
                pages.append({"source": name, "page": i + 1, "text": page.get_text()})
    return pages

pages = ingest(PAPER_DIR)
print(pd.DataFrame(pages).groupby("source").agg(pages=("page", "max"), chars=("text", lambda s: s.str.len().sum())).to_string())
print(f"\nTotal: {len(pages)} pages")

## 1.2 Inspect the raw text

Two common problems: **broken hyphenation** (`transduc-\ntion`) and **ligatures** (`ﬁ` is one Unicode character, U+FB01). `normalize_text` drops non-ASCII characters, so `speciﬁc` becomes `speci c` and `ﬁne-tuning` becomes `ne tuning`.

In [ ]:
raw_all = "\n".join(p["text"] for p in pages)
LIGATURES = {"ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "st", "ﬆ": "st"}
HYPHEN_BREAK = re.compile(r"([A-Za-z]+)-\n([A-Za-z]+)")

lig_counts = Counter(c for c in raw_all if c in LIGATURES)
print("Ligature characters:", {f"{k} (U+{ord(k):04X})": v for k, v in lig_counts.items()})
print("Hyphenated line breaks:", len(HYPHEN_BREAK.findall(raw_all)))
print("  e.g.", [f"{a}-{b}" for a, b in HYPHEN_BREAK.findall(raw_all)[:10]])

for word in ["speciﬁc", "ﬁne-tuning", "transduc-\ntion"]:
    i = raw_all.find(word)
    print(f"\nRaw:        {raw_all[max(0, i - 50):i + 40]!r}")
    print(f"Normalized: {normalize_text(raw_all[max(0, i - 50):i + 40])!r}")

**Do the 12 reference answers survive extraction?** Two checks, each run against the answer's own paper:
- **strict**: the exact answer string appears in the text, with whitespace collapsed. A line break inside the phrase is fine, a broken hyphen or ligature is not.
- **normalized**: the answer appears after `normalize_text` on both sides. This is what `hit@k` uses.

In [ ]:
def paper_text(pages, src, transform=lambda t: t):
    return "\n".join(transform(p["text"]) for p in pages if p["source"] == src)

def count_occurrences(answer, text):
    pat = r"(?<![a-z0-9])" + re.escape(normalize_text(answer)) + r"(?![a-z0-9])"
    return len(re.findall(pat, normalize_text(text)))

def survival(pages, qs, transform=lambda t: t):
    rows = []
    for q in qs:
        text = paper_text(pages, q["paper"], transform)
        rows.append({
            "qid": q["qid"], "answer": q["answer"], "paper": q["paper"],
            "strict": int(q["answer"] in " ".join(text.split())),
            "normalized": int(count_occurrences(q["answer"], text) > 0),
            "occurrences": count_occurrences(q["answer"], text),
        })
    return pd.DataFrame(rows)

surv_raw = survival(pages, questions)
print(surv_raw.to_string(index=False))
print(f"\nRaw text: strict {surv_raw.strict.sum()}/12 | normalized {surv_raw.normalized.sum()}/12")

## 1.3 Clean and measure again

- **Ligatures**: replace `ﬁ ﬂ ﬀ ﬃ ﬄ` with their letters.
- **Hyphenation**: for each `word-\nword` line break:
  - if the hyphenated form appears elsewhere on one line (`pre-training`), keep the hyphen
  - else if the joined word appears elsewhere (`transduction`), join it
  - otherwise keep the hyphen

In [ ]:
def fix_ligatures(text: str) -> str:
    for lig, letters in LIGATURES.items():
        text = text.replace(lig, letters)
    return text

VOCAB = {w.lower() for w in re.findall(r"[A-Za-z]+", fix_ligatures(raw_all))}
HYPHENATED = {w.lower() for w in re.findall(r"[A-Za-z]+-[A-Za-z]+", fix_ligatures(raw_all))}   # e.g. pre-training, on one line

def dehyphenate(text: str) -> str:
    def join(m):
        a, b = m.group(1), m.group(2)
        if f"{a}-{b}".lower() in HYPHENATED:   # a real compound: keep the hyphen
            return f"{a}-{b}"
        return a + b if (a + b).lower() in VOCAB else f"{a}-{b}"
    return HYPHEN_BREAK.sub(join, text)

def clean_text(text: str) -> str:
    return dehyphenate(fix_ligatures(text))

clean_all = "\n".join(clean_text(p["text"]) for p in pages)
print("Ligatures left:", sum(clean_all.count(c) for c in LIGATURES))
print("Hyphenated line breaks left:", len(HYPHEN_BREAK.findall(clean_all)), HYPHEN_BREAK.findall(clean_all)[:6])

surv_clean = survival(pages, questions, clean_text)
before_after = surv_raw[["qid", "answer", "paper"]].assign(
    strict_raw=surv_raw.strict, strict_clean=surv_clean.strict,
    norm_raw=surv_raw.normalized, norm_clean=surv_clean.normalized,
    occ_raw=surv_raw.occurrences, occ_clean=surv_clean.occurrences)
print(before_after.to_string(index=False))
print(f"\nAnswers found, strict:     raw {surv_raw.strict.sum()}/12 -> clean {surv_clean.strict.sum()}/12")
print(f"Answers found, normalized: raw {surv_raw.normalized.sum()}/12 -> clean {surv_clean.normalized.sum()}/12")

The answers are short and mostly survive. Cleaning matters more for the **question words** that retrieval has to match. Below: how often each content word of the questions appears in the corpus before and after cleaning.

In [ ]:
STOP = set("what which when does how many make used uses that this from with into the and are for its by".split())
words = sorted({w for q in QS for w in normalize_text(q["question"]).split() if len(w) > 3 and w not in STOP})
word_counts = pd.DataFrame([{"word": w, "raw": count_occurrences(w, raw_all), "clean": count_occurrences(w, clean_all)} for w in words])
changed = word_counts[word_counts.raw != word_counts.clean].sort_values("clean", ascending=False)
print(changed.to_string(index=False))

survival(pages, own_questions, clean_text)   # our own answers must be in the cleaned text

## 1.4 Chunk

`RecursiveCharacterTextSplitter` from 6.6 at **300** and **800** characters, on both **raw** and **cleaned** text, so we can measure the effect of cleaning too. Each paper is split as one document, so chunks can cross page breaks. `add_start_index` tells us which page each chunk starts on.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_docs(pages, cleaned: bool) -> list[dict]:
    docs = []
    for src in PAPERS:
        text, page_starts = "", []
        for p in pages:
            if p["source"] == src:
                page_starts.append(len(text))
                text += (clean_text(p["text"]) if cleaned else p["text"]) + "\n\n"
        docs.append({"source": src, "text": text, "page_starts": page_starts})
    return docs

def chunk(docs, size: int, overlap: int) -> list[dict]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap,
                                              separators=["\n\n", "\n", ". ", " ", ""], add_start_index=True)
    chunks = []
    for d in docs:
        for c in splitter.create_documents([d["text"]]):
            page = bisect.bisect_right(d["page_starts"], c.metadata["start_index"])   # 1-based page number
            chunks.append({"id": len(chunks), "source": d["source"], "page": page, "text": c.page_content})
    return chunks

CHUNKINGS = {   # name -> (cleaned?, chunk_size, overlap)
    "raw-300":   (False, 300, 50),
    "raw-800":   (False, 800, 100),
    "clean-300": (True, 300, 50),
    "clean-800": (True, 800, 100),
}
CHUNKS = {name: chunk(build_docs(pages, cleaned), size, ov) for name, (cleaned, size, ov) in CHUNKINGS.items()}
for name, ch in CHUNKS.items():
    sizes = [len(c["text"]) for c in ch]
    print(f"{name:<10} {len(ch):>5} chunks | mean {np.mean(sizes):>4.0f} chars | per paper {dict(Counter(c['source'] for c in ch))}")
print("\nExample:", CHUNKS["clean-300"][40])

## 1.5 Index: BM25 and dense

BM25 with `rank_bm25` (6.7). Dense vectors from the config model, held in memory as a NumPy matrix (fine at this size). Chunk embeddings are saved to the results folder, so a Colab restart does not recompute them.

BGE models expect a short instruction in front of **questions** (not passages). We add it to questions only. A HyDE hypothesis is passage-like, so it is embedded without the instruction.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(DENSE_MODEL, device=device)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: " if "bge" in DENSE_MODEL.lower() else ""

def tokenize(text):
    return re.findall(r"\w+", text.lower())

def embed_chunks(name, chunks):
    path = RESULTS_DIR / f"emb_{name}_{DENSE_MODEL.split('/')[-1]}.npy"
    if path.exists():
        emb = np.load(path)
        if len(emb) == len(chunks):
            print(f"{name}: loaded embeddings from {path.name}")
            return emb
    emb = embedder.encode([c["text"] for c in chunks], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    np.save(path, emb)
    return emb

INDEXES = {name: {"chunks": ch,
                  "bm25": BM25Okapi([tokenize(c["text"]) for c in ch]),
                  "emb": embed_chunks(name, ch)}
           for name, ch in CHUNKS.items()}
print("Indexed:", {n: idx["emb"].shape for n, idx in INDEXES.items()})

---
# Part 2: Retrieval and the full pipeline

## 2.1 Retrievers

BM25, dense, and hybrid with `reciprocal_rank_fusion`, copied from 6.7. Hybrid fuses the top-20 of each list, as in 6.7 and 6.13. All of them return chunk ids, best first.

In [ ]:
_qcache = {}
def encode_query(text, prefix):
    if (prefix, text) not in _qcache:
        _qcache[(prefix, text)] = embedder.encode(prefix + text, normalize_embeddings=True)
    return _qcache[(prefix, text)]

def rank_bm25(idx, query, k=5):
    return [int(i) for i in np.argsort(-idx["bm25"].get_scores(tokenize(query)))[:k]]

def rank_dense(idx, query, k=5, prefix=None):
    q = encode_query(query, QUERY_PREFIX if prefix is None else prefix)
    return [int(i) for i in np.argsort(-(idx["emb"] @ q))[:k]]

def reciprocal_rank_fusion(bm25_ranks, dense_ranks, k=60, top_n=5):   # from 6.7
    rrf_scores = {}
    for ranking in (bm25_ranks, dense_ranks):
        for rank_pos, doc_id in enumerate(ranking, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank_pos)
    sorted_docs = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    return [d for d, _ in sorted_docs[:top_n]], [s for _, s in sorted_docs[:top_n]]

def rank_hybrid(idx, query, k=5, pool=20, dense_query=None, dense_prefix=None):
    # dense_query lets HyDE send the hypothesis to the dense side while BM25 keeps the question
    fused, _ = reciprocal_rank_fusion(rank_bm25(idx, query, pool),
                                      rank_dense(idx, dense_query or query, pool, dense_prefix), top_n=k)
    return fused

RETRIEVERS = {"BM25": rank_bm25, "Dense": rank_dense, "Hybrid RRF": rank_hybrid}

## 2.2 hit@1, hit@3, hit@5

For each question we store the rank of the first chunk that contains the answer (`None` if it is not in the top 5). Then hit@k is the share of questions with that rank ≤ k.

In [ ]:
K_MAX = 5

def contains_answer(chunk, answer):
    return normalize_text(answer) in normalize_text(chunk["text"])

def first_hit(idx, ranked, answer):
    for r, i in enumerate(ranked[:K_MAX], start=1):
        if contains_answer(idx["chunks"][i], answer):
            return r
    return None

def evaluate(retrievers: dict) -> pd.DataFrame:
    rows = []
    for cname, idx in INDEXES.items():
        for rname, fn in retrievers.items():
            for q in QS:
                rows.append({"chunking": cname, "retriever": rname, "qid": q["qid"], "paper": q["paper"],
                             "own": q["own"], "rank": first_hit(idx, fn(idx, q), q["answer"])})
    return pd.DataFrame(rows)

def hitk_table(df):
    t = df.assign(**{f"hit@{k}": df["rank"].le(k) for k in (1, 3, 5)})   # rank None -> NaN -> False
    return t.groupby(["chunking", "retriever"], sort=False)[["hit@1", "hit@3", "hit@5"]].mean().round(2)

def per_paper(df, k=3):
    return df.assign(hit=df["rank"].le(k)).pivot_table(index=["chunking", "retriever"], columns="paper",
                                                         values="hit", aggfunc="mean", sort=False).round(2)

# retrievers get a question dict; plain retrievers use the question text
base_runs = evaluate({name: (lambda idx, q, fn=fn: fn(idx, q["question"], K_MAX)) for name, fn in RETRIEVERS.items()})
print(f"hit@k on all {len(QS)} questions\n")
print(hitk_table(base_runs).to_string())

In [ ]:
print("hit@k on the 12 given questions only\n")
print(hitk_table(base_runs[~base_runs.own]).to_string())
print("\nhit@3 per paper (all questions)\n")
print(per_paper(base_runs).to_string())

best = hitk_table(base_runs).mean(axis=1).idxmax()
print(f"\nBest combination (mean of hit@1/3/5): {best}")

**Which questions does each retriever miss?** Rank of the first answer-bearing chunk on the cleaned 300-character index (`-` = not in top 5).

In [ ]:
def rank_grid(df, chunking):
    g = df[df.chunking == chunking].pivot(index="qid", columns="retriever", values="rank")
    g = g.apply(lambda col: col.map(lambda r: "-" if pd.isna(r) else int(r)))
    info = pd.DataFrame(QS).set_index("qid")[["paper", "answer"]]
    return info.join(g)

print(rank_grid(base_runs, "clean-300").to_string())

## 2.3 Exact-term query vs. paraphrase

- `WMT 2014 BLEU 28.4`: target = a chunk containing `28.4`
- `dispensing with sequential recurrence in sequence modeling`: target = the abstract, which says the Transformer is *"dispensing with recurrence and convolutions entirely"*. The query shares `dispensing` and `recurrence` with it, so we also try a paraphrase with **no shared content words**.

In [ ]:
PROBES = [
    ("WMT 2014 BLEU 28.4", "28.4"),
    ("dispensing with sequential recurrence in sequence modeling", "dispensing with recurrence"),
    ("a translation model that drops RNNs and relies only on attention", "dispensing with recurrence"),
]
idx = INDEXES["clean-300"]
for query, target in PROBES:
    print(f"\nQuery: {query!r}   (target text: {target!r})")
    for rname, fn in RETRIEVERS.items():
        ranked = fn(idx, query, 20)
        r = next((i for i, cid in enumerate(ranked, 1) if normalize_text(target) in normalize_text(idx["chunks"][cid]["text"])), None)
        top = idx["chunks"][ranked[0]]
        print(f"  {rname:<11} target rank: {r if r else '>20':>4} | top-1 ({top['source']} p.{top['page']}): {' '.join(top['text'].split())[:90]}")

**Why** (from our run on the local PDFs; check the printed ranks):
- **Exact-term query: all three tie at rank 1.** `28.4`, `WMT`, `2014` and `BLEU` all sit in the same abstract sentence. BM25 matches the rare tokens exactly. Dense search also works because the query and the sentence are near copies of each other.
- **Given paraphrase: BM25 wins (rank 2), dense misses (rank > 20).** The "paraphrase" still contains `dispensing` and `recurrence`. `dispensing` occurs only in the abstract, so its BM25 weight (IDF) is high. The dense model sees a short chunk about "attention mechanisms", like hundreds of others in the paper, so nothing makes it stand out.
- **Paraphrase with no shared words: everyone fails.** Hybrid cannot rescue a chunk that neither list ranks high. That is the case HyDE is meant for: the LLM writes a passage in the paper's own vocabulary.
- So dense search does not win paraphrases by default. A small embedding model on a corpus where every chunk is about attention and sequence models has little room to separate chunks by meaning.

## 2.4 LLM setup (needed for HyDE and cited answers)

Two ways to get an LLM:
1. **OpenRouter** (or any OpenAI-compatible API). In Colab, add a secret named `OPENROUTER_API_KEY` (key icon in the left bar) and allow this notebook to use it.
2. **Ollama inside Colab.** With no key, the cell installs Ollama, starts it and pulls `llama3.2:3b` (the config.yaml model). This needs a GPU runtime and takes about 2 minutes.

Locally, it uses Ollama on `localhost:11434`, as in the course notebooks. Every reply is cached in `llm_cache.json` on Drive, so reruns make no new LLM calls.

In [ ]:
from langchain_openai import ChatOpenAI

OLLAMA_MODEL = "llama3.2:3b"
OPENROUTER_MODEL = "meta-llama/llama-3.2-3b-instruct"   # any OpenRouter model id works

def get_secret(name):
    if os.getenv(name):
        return os.getenv(name)
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None

def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
        return True
    except Exception:
        return False

key = get_secret("OPENROUTER_API_KEY")
if key:
    LLM_NAME = OPENROUTER_MODEL
    llm = ChatOpenAI(model=LLM_NAME, base_url="https://openrouter.ai/api/v1", api_key=key, temperature=0)
else:
    if IN_COLAB and shutil.which("ollama") is None:
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)
    if not ollama_up() and shutil.which("ollama"):
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            if ollama_up():
                break
            time.sleep(1)
    if IN_COLAB:
        subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True, stdout=subprocess.DEVNULL)
    LLM_NAME = OLLAMA_MODEL
    llm = ChatOpenAI(model=LLM_NAME, base_url="http://localhost:11434/v1", api_key="ollama", temperature=0)

print(f"LLM: {LLM_NAME} ->", llm.invoke("Reply with OK.").content)

In [ ]:
LLM_CACHE_FILE = RESULTS_DIR / "llm_cache.json"
RESUME = True   # set False to regenerate every LLM reply
llm_cache = json.loads(LLM_CACHE_FILE.read_text()) if RESUME and LLM_CACHE_FILE.exists() else {}
print(f"{len(llm_cache)} cached LLM replies in {LLM_CACHE_FILE}")

def ask_llm(task, prompt):
    # One call per (model, task, prompt); saved to Drive straight away
    key = f"{LLM_NAME}|{task}|{hashlib.sha1(prompt.encode()).hexdigest()[:16]}"
    if key not in llm_cache:
        llm_cache[key] = llm.invoke(prompt).content
        LLM_CACHE_FILE.write_text(json.dumps(llm_cache, indent=1))
    return llm_cache[key]

## 2.5 HyDE

This is the 6.8 HyDE prompt, with *"Wikipedia excerpt"* changed to *"research paper excerpt"* to suit this corpus. The LLM writes a hypothetical answer passage, and we embed that instead of the question. One hypothesis is made per question and reused for every chunking.
- **Dense + HyDE**: dense search with the hypothesis.
- **Hybrid + HyDE**: BM25 with the question, dense with the hypothesis, then RRF.

In [ ]:
HYDE_PROMPT = '''
Write a short factual passage (2-3 sentences) that would directly answer the following question.
Do NOT say you don't know — write your best guess as if it were an excerpt from a research paper.

Question: {question}

Passage:
'''

for q in QS:
    q["hyde"] = ask_llm("hyde", HYDE_PROMPT.format(question=q["question"])).strip()
print(QS[0]["question"], "\n->", QS[0]["hyde"])

hyde_runs = evaluate({
    "Dense + HyDE":  lambda idx, q: rank_dense(idx, q["hyde"], K_MAX, prefix=""),
    "Hybrid + HyDE": lambda idx, q: rank_hybrid(idx, q["question"], K_MAX, dense_query=q["hyde"], dense_prefix=""),
})
all_runs = pd.concat([base_runs, hyde_runs], ignore_index=True)
order = ["BM25", "Dense", "Dense + HyDE", "Hybrid RRF", "Hybrid + HyDE"]
table = hitk_table(all_runs).reindex([(c, r) for c in CHUNKINGS for r in order])
print(f"\nhit@k on all {len(QS)} questions\n")
print(table.to_string())

**Where does HyDE help?** First-hit rank with and without HyDE (`-` = not in the top 5), on the chunking where dense search did best.

In [ ]:
best_dense_chunking = hitk_table(base_runs).xs("Dense", level="retriever").mean(axis=1).idxmax()
grid = rank_grid(all_runs[all_runs.retriever.isin(order)], best_dense_chunking)[["paper", "answer", *order]]
print(f"Chunking: {best_dense_chunking}\n")
print(grid.to_string())

def better(a, b):   # True if rank a beats rank b
    a, b = (99 if x == "-" else x for x in (a, b))
    return a < b
for base, hy in [("Dense", "Dense + HyDE"), ("Hybrid RRF", "Hybrid + HyDE")]:
    up = [qid for qid, r in grid.iterrows() if better(r[hy], r[base])]
    down = [qid for qid, r in grid.iterrows() if better(r[base], r[hy])]
    print(f"\n{hy}: better on {up} | worse on {down}")
print("\nHypotheses for the questions HyDE changed most:")
for q in QS:
    if q["qid"] in grid.index and grid.loc[q["qid"], "Dense"] != grid.loc[q["qid"], "Dense + HyDE"]:
        print(f"  {q['qid']} {q['question']}\n      HyDE: {' '.join(q['hyde'].split())[:200]}")

## 2.6 Cited answers (6.13)

The model sees the **top-5 hybrid chunks**, numbered, from the best chunking. It returns JSON with the answer and the numbers of the chunks it used. The prompt and parser are copied from 6.13.

We score:
- **context hit**: the reference answer is in one of the 5 chunks
- **cited correct**: at least one cited chunk contains the reference answer
- **citation precision**: the share of cited chunks that contain it
- **answer F1**: token F1 against the reference

In [ ]:
ANSWER_PROMPT = '''Answer the question using ONLY the numbered passages below.
Reply with JSON only, in this exact form: {{"answer": "<short answer>", "sources": [<numbers of the passages you used>]}}
If the passages do not contain the answer, reply {{"answer": "I don't know", "sources": []}}.

{passages}

Question: {question}
JSON:'''

def parse_json(text):   # from 6.13
    match = re.search(r"\{.*\}", text, re.S)
    try:
        data = json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        data = {}
    answer = str(data.get("answer", text)).strip()
    sources = [int(s) for s in data.get("sources", []) if str(s).isdigit()]
    return answer, sources

CITE_CHUNKING = best[0]   # best = (chunking, retriever) from 2.2
idx = INDEXES[CITE_CHUNKING]
cite_rows = []
for q in QS:
    ids = rank_hybrid(idx, q["question"], K_MAX)
    numbered = "\n\n".join(f"[{i}] {idx['chunks'][cid]['text']}" for i, cid in enumerate(ids, start=1))
    answer, sources = parse_json(ask_llm("cite", ANSWER_PROMPT.format(passages=numbered, question=q["question"])))
    cited = [ids[s - 1] for s in sources if 1 <= s <= len(ids)]   # ignore invented numbers
    good = [cid for cid in cited if contains_answer(idx["chunks"][cid], q["answer"])]
    cite_rows.append({
        "qid": q["qid"], "paper": q["paper"], "reference": q["answer"], "prediction": answer,
        "cited": [f"{idx['chunks'][c]['source'].split('.')[0]} p.{idx['chunks'][c]['page']}" for c in cited],
        "context_hit": int(any(contains_answer(idx["chunks"][c], q["answer"]) for c in ids)),
        "cited_correct": int(bool(good)),
        "cite_precision": len(good) / len(cited) if cited else 0.0,
        "f1": round(token_f1(answer, q["answer"]), 2),
    })
cites = pd.DataFrame(cite_rows)
print(f"Chunking: {CITE_CHUNKING} | retriever: Hybrid RRF top-{K_MAX} | LLM: {LLM_NAME}\n")
print(cites.to_string(index=False))

n, n_ctx = len(cites), cites.context_hit.sum()
print(f"\nAnswer in the 5 chunks (context hit):      {n_ctx}/{n}")
print(f"Cited a chunk that contains the answer:    {cites.cited_correct.sum()}/{n}  "
      f"({cites[cites.context_hit == 1].cited_correct.sum()}/{n_ctx} when the answer was in context)")
print(f"Citation precision (mean):                 {cites.cite_precision.mean():.2f}")
print(f"Answer token F1 (mean):                    {cites.f1.mean():.2f}")

## 2.7 Save the results

The tables go to the Drive results folder, so they survive the Colab session.

In [ ]:
runs = all_runs if "all_runs" in globals() else base_runs
runs.to_csv(RESULTS_DIR / "hitk_ranks.csv", index=False)
final = hitk_table(runs)
final.to_csv(RESULTS_DIR / "hitk_table.csv")
before_after.to_csv(RESULTS_DIR / "cleaning_before_after.csv", index=False)
parts = ["# Exercise 6.1 results", f"Dense model: {DENSE_MODEL}",
         "## hit@k (all questions)", final.to_markdown(),
         "## hit@3 per paper", per_paper(runs).to_markdown(),
         "## Cleaning: reference answers found (12 given questions)", before_after.to_markdown(index=False)]
if "cites" in globals():
    cites.to_csv(RESULTS_DIR / "cited_answers.csv", index=False)
    parts += [f"## Cited answers ({CITE_CHUNKING}, hybrid top-{K_MAX}, {LLM_NAME})", cites.to_markdown(index=False)]
(RESULTS_DIR / "results.md").write_text("\n\n".join(parts))
print("Saved to", RESULTS_DIR.resolve(), sorted(p.name for p in RESULTS_DIR.iterdir()))

---
# Part 3: Analysis

The facts below feed the five sentences. Rerun this cell after the LLM cells, so the HyDE rows are included.

In [ ]:
runs = all_runs if "all_runs" in globals() else base_runs
h = hitk_table(runs)
print("hit@3 per paper, clean-300 vs clean-800:\n")
print(per_paper(runs[runs.chunking.isin(["clean-300", "clean-800"])]).to_string())
print("\nMean hit@k by chunking (all retrievers):\n", h.groupby(level="chunking", sort=False).mean().round(2).to_string())
print("\nMean hit@k by retriever (all chunkings):\n", h.groupby(level="retriever", sort=False).mean().round(2).to_string())

### Five sentences

_These numbers come from a local run on the PDFs in `Week6/data/papers/`, using `bge-small-en-v1.5`. If your Colab tables differ (for example, because the PDFs were downloaded fresh from arXiv), update the numbers._

1. **Retriever by paper.** Dense search is strongest on the Attention paper (hit@5 = 1.00 with 800-character chunks), because those questions reuse the paper's own phrases ("positional encodings", "multi-head attention") and the paper is short. BM25 is strongest on BERT (0.67 vs 0.33–0.50 for dense), because the BERT questions hinge on specific terms ("[MASK]", "feature-based", "WordPiece"), while to a dense model dozens of BERT chunks about pre-training and fine-tuning look the same.
2. **GPT-3 depends on chunk size.** With 300-character chunks, dense beats BM25 on GPT-3 (0.8 vs 0.4 hit@5), because in a 75-page paper common question words like "model" and "learning" appear in hundreds of chunks. With 800-character chunks, BM25 catches up (0.8–1.0), because a longer chunk holds the rare terms ("Common Crawl", "learning rate") in the same place as the answer.
3. **Chunk size mattered more than cleaning.** Going from 300 to 800 characters raised mean hit@5 by about 0.1 (0.63–0.67 → 0.75–0.78). Cleaning moved it by 0.04 or less, in either direction, because all 12 answers already survived extraction and `normalize_text` already repairs most line-break damage. The ligature fix did turn 25 matches of "fine" into 169, but few of our questions depend on such words.
4. **Retriever choice mattered about as much as chunk size.** Hybrid RRF had the best mean hit@5 (0.76, vs 0.69 for BM25 and 0.66 for dense) and was best or tied in every chunking at hit@5. At hit@1 it rarely beat the better single retriever: fusion mostly helps recall, not the top spot.
5. **Hybrid helps most on GPT-3.** With short chunks it rescues the GPT-3 questions BM25 misses (0.4 → 0.8 hit@5) and adds almost nothing on Attention, where both retrievers already find the same chunks. With only 5–6 questions per paper, one question moves a score by 0.17–0.2, so smaller differences are noise.

_After running 2.5–2.6, add one line on HyDE (which questions it rescued or broke) and the citation rate._